In [8]:
import torch 

if torch.cuda.is_available():
    device=torch.device="cuda"
else:
    device=torch.device="cpu"

print(device)

cuda


In [9]:
my_tensor=torch.randn(3,requires_grad=True)

x1=my_tensor+my_tensor
x1

tensor([ 1.0596, -2.2291,  0.4081], grad_fn=<AddBackward0>)

In [10]:
x2=x1*x1
x2

tensor([1.1228, 4.9690, 0.1665], grad_fn=<MulBackward0>)

In [12]:
x2=x2.mean()
x2

tensor(2.0861, grad_fn=<MeanBackward0>)

In [13]:
x2.backward()
print(my_tensor.grad)

tensor([ 1.4128, -2.9722,  0.5441])


# if the final output not a scalar

In [14]:
my_tensor2=torch.randn(3,requires_grad=True)

x11=my_tensor2+my_tensor2
x11

tensor([ 2.1566,  0.6495, -3.0958], grad_fn=<AddBackward0>)

In [15]:
x12=x11*x11
x12

tensor([4.6508, 0.4218, 9.5837], grad_fn=<MulBackward0>)

# 🧠 فهم كيفية عمل `backward()` مع الـ Vectors في PyTorch

في PyTorch، دالة `backward()` مصممة بشكل افتراضي للعمل على القيمة الفردية (Scalar)، لأن الهدف الأساسي للشبكات العصبية هو تقليل **دالة التكلفة (Loss Function)** والتي تكون دائماً رقماً واحداً يعبر عن إجمالي الخطأ.

عند محاولة استخدام `backward()` على تنسور يحتوي على أكثر من قيمة (Vector)، سيظهر خطأ. السبب الرياضي لذلك هو أننا نحتاج لتحديد **"وزن"** أو **"أهمية"** لكل عنصر في هذا الـ Vector أثناء حساب التفاضل (Gradients).

---

### 🧮 مثال رياضي مبسط

لنفترض أن لدينا المتغير $x$ يحتوي على رقمين، وقمنا بتربيعه لنحصل على $y$:

* المتغير الأساسي: $x = [x_1, x_2] = [2, 3]$
* العملية الحسابية: $y = x^2$
* النتيجة: $y = [y_1, y_2] = [4, 9]$

رياضياً، تفاضل $x^2$ هو $2x$. أي أن التفاضل الأساسي هو:
* تفاضل $y_1$ بالنسبة لـ $x_1$ هو: $2 \times 2 = 4$
* تفاضل $y_2$ بالنسبة لـ $x_2$ هو: $2 \times 3 = 6$

---

### ❓ لماذا يطلب PyTorch تمرير متّجه (Vector `v`)؟

عندما نستدعي `y.backward()`، فإن PyTorch يتساءل: **"كيف أوزع قيمة الخطأ على العنصرين $y_1$ و $y_2$؟"**.
لحل هذه المشكلة، يجب أن نمرر Vector $v$ بنفس حجم $y$ ليمثل **الأوزان (Weights)**.

المعادلة التي ينفذها PyTorch هي:
$$\text{Gradient} = \left( \frac{\partial y}{\partial x} \times v \right)$$

#### 💡 سيناريوهات تمرير الـ Vector:

1. **إعطاء أهمية متساوية للجميع:**
   إذا مررنا متّجه من الواحد الصحيح $v = [1, 1]$:
   * للعنصر الأول: $4 \times 1 = 4$
   * للعنصر الثاني: $6 \times 1 = 6$
   * **النتيجة النهائية (`x.grad`):** ستكون `[4, 6]`

2. **إعطاء أهمية لعنصر وتجاهل الآخر:**
   إذا مررنا متّجه بأوزان مختلفة $v = [10, 0]$:
   * للعنصر الأول: $4 \times 10 = 40$
   * للعنصر الثاني: $6 \times 0 = 0$
   * **النتيجة النهائية (`x.grad`):** ستكون `[40, 0]`

---

### 💻 كيفية حل المشكلة برمجياً

عندما تقوم باختبار العمليات الرياضية وليس لديك `Loss Function` حقيقية، يمكنك حل المشكلة بإحدى الطريقتين:

**الطريقة الأولى: تمرير Vector من الواحد الصحيح (Ones Vector)**
```python
import torch

# إعطاء أهمية متساوية لكل العناصر
v = torch.ones_like(y) 
y.backward(v)

In [ ]:
#! we will found the error message 

# x12.backward()
# print(my_tensor2.grad)

In [ ]:
# we must make a vector same size like output and pass to backward(v)

v=torch.ones_like(x12)
x12.backward(v)
print(my_tensor2.grad)

tensor([  8.6263,   2.5980, -12.3830])


# when you want to deactive requires grad in some places 

In [27]:
# reqires_grad_(False)
# .detach()
# with torch.no_grad():

x22=torch.randn(3,requires_grad=True)
x33=torch.randn(3,requires_grad=True)
x44=torch.randn(3,requires_grad=True)

In [ ]:
x22.requires_grad_(False)


tensor([-0.4289,  0.8300,  0.0141])

In [29]:
g=x33.detach()
g

tensor([0.2958, 0.3091, 1.1656])

In [ ]:
with torch.no_grad():
    x44=x44*2
    print(x44)

tensor([ 1.4669, -1.2795,  6.4320])

# while tranning we must empty the grad every iteration

### 🔄 لماذا تتراكم التدرجات (Gradient Accumulation) في PyTorch؟

سلوك تراكم التدرجات **ليس عيباً برمجياً (Bug)**، بل هو ميزة هندسية ورياضية مقصودة لتسهيل عملية التدريب.

**📌 النقاط الرئيسية:**
* **الأساس الرياضي:** يعتبر هذا السلوك تطبيقاً مباشراً لـ **قاعدة السلسلة (Chain Rule)** في التفاضل.
* **الفائدة العملية:** يُسهّل جداً تدريب الهيكليات المعقدة التي تعتمد على "مشاركة الأوزان" (مثل شبكات **RNNs**)، حيث يقوم PyTorch بجمع التفاضلات الناتجة من خطوات مختلفة لنفس الوزن تلقائياً، دون الحاجة لكتابة كود معقد لجمعها يدوياً.
* **مسؤولية المبرمج:** نظراً لأن PyTorch يجمع التدرجات فوق بعضها بشكل افتراضي، **يجب عليك إجبارياً** تصفير هذه الذاكرة قبل بدء دورة تدريب جديدة (Batch أو Epoch جديد) لضمان عدم تداخل حسابات الماضي مع الحاضر.

> **💡 الخلاصة البرمجية:**
> لا تنسَ أبداً استخدام دالة التصفير في بداية كل حلقة تدريب:
```python
# إذا كنت تستخدم Optimizer
optimizer.zero_grad()

# إذا كنت تختبر تنسور معين يدوياً
# weights.grad.zero_()

In [ ]:
weights=torch.ones(3,requires_grad=True)
for epoch in range(5):
#! without empty the grad
    model_ouput=(weights*3).sum()
    model_ouput.backward()
    print(weights.grad)


tensor([3., 3., 3.])
tensor([6., 6., 6.])
tensor([9., 9., 9.])
tensor([12., 12., 12.])
tensor([15., 15., 15.])


In [34]:
weights2=torch.ones(3,requires_grad=True)
for epoch in range(5):
#! without empty the grad
    model_ouput2=(weights2*3).sum()
    model_ouput2.backward()
    print(weights2.grad)
    weights2.grad.zero_()

tensor([3., 3., 3.])
tensor([3., 3., 3.])
tensor([3., 3., 3.])
tensor([3., 3., 3.])
tensor([3., 3., 3.])


### 🔄 الترتيب الصحيح لخطوات التدريب باستخدام Optimizer

عند استخدام مُحسّن الأوزان (Optimizer) مثل `SGD` أو `Adam`، هناك ترتيب منطقي لـ 4 خطوات أساسية تتكرر في كل لفة (Iteration/Epoch):

```python
# تهيئة المُحسّن وتمرير الأوزان له في قائمة
optimizer = torch.optim.SGD([weights], lr=0.01)

for epoch in range(5):
    # 1. تصفير التدرجات: نمسح تدرجات اللفة السابقة حتى لا تتراكم
    optimizer.zero_grad()
    
    # 2. المرور الأمامي (Forward Pass): إجراء العمليات الحسابية
    model_output = (weights * 3).sum()
    
    # 3. المرور الخلفي (Backward Pass): حساب التدرجات الجديدة
    model_output.backward()
    
    # 4. تحديث الأوزان (Optimizer Step): تعديل الأوزان بناءً على التدرجات المحسوبة
    optimizer.step()

In [ ]:
#! we can do the same thing with optimizer 
optimizer=torch.optim.SGD([weights] ,lr=0.01)
optimizer.step()
optimizer.zero_grad()

: 